<a href="https://colab.research.google.com/github/DARIAH-SI/Brokers-of-NAM/blob/main/Entity_normalization_and_CSVs_from_JSON.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
#This script:
# i. extracts the entities (PER, LOC, ORG) from a JSON file with named entities from sources in Serbo-Croatian;
# ii. establishes normalized forms in English for the named entities via DeepSeek's API;
# iii. Downloads one CSV by category of entity.

#The script was largely developed with the help of Claude's Sonnet 4.6.
#The prompt for DeepSeek's API is in French.

import re
import csv
import json
import time
from getpass import getpass
from pathlib import Path
from collections import defaultdict
from google.colab import files
from openai import OpenAI


# ── API KEY ───────────────────────────────────────────────────────────────────

DEEPSEEK_API_KEY = getpass("Clé API DeepSeek : ")
client = OpenAI(api_key=DEEPSEEK_API_KEY, base_url="https://api.deepseek.com")
print("✓ API configurée")


# ── Prompt ────────────────────────────────────────────────────────────────────

DEDUP_SYSTEM = (
    "Tu es un expert en morphologie serbo-croate et en histoire yougoslave. "
    "Tu retournes UNIQUEMENT du JSON valide, sans texte autour, sans balises markdown."
)

DEDUP_PROMPT = """Voici une liste d'entités de type {category} extraites d'un texte serbo-croate.

Le serbo-croate est une langue à déclinaisons. Des formes comme :
  Kardelj / Kardelja / Kardelju désignent la même personne
  Socijalističkog saveza / Socijalističkom savezu / Socijalistički savez désignent la même organisation

Le texte mentionne aussi des entités étrangères (organisations internationales, partis politiques,
institutions d'autres pays). Pour ces entités, le nom canonique doit être dans la langue d'origine
de l'organisation ou du lieu, et non sa forme serbo-croate déclinée.

Exemples :
  "Laburističke partije" → "Labour Party"  (parti britannique, nom en anglais)
  "Socijaldemokratske partije Zapadne Njemačke" → "Sozialdemokratische Partei Deutschlands"  (SPD)
  "Komunističke partije Francuske" → "Parti communiste français"
  "Ujedinjenih nacija" → "United Nations"
  "Socijalistički savez radnog naroda Jugoslavije" → "Socijalistički savez radnog naroda Jugoslavije"
    (organisation yougoslave, nom canonique en serbo-croate au nominatif)

Regroupe les entrées qui désignent clairement la même entité.
Pour chaque groupe, choisis la forme canonique selon ces règles :
  - Entité yougoslave / serbo-croate → nominatif complet en serbo-croate
  - Entité étrangère → nom officiel dans la langue d'origine de l'organisation ou du pays
Garde TOUTES les entités, même celles sans variantes (liste vide).

Retourne UNIQUEMENT un objet JSON valide :
{{
  "entites": [
    {{
      "nom_canonique": "Labour Party",
      "variantes": ["Laburističke partije", "Laburističkoj partiji"]
    }},
    {{
      "nom_canonique": "Socijalistički savez radnog naroda Jugoslavije",
      "variantes": ["Socijalističkog saveza radnog naroda Jugoslavije", "Socijalističkom savezu"]
    }},
    {{
      "nom_canonique": "Beograd",
      "variantes": []
    }}
  ]
}}

Entités à traiter :
{entities}
"""

TYPE_LABELS = {
    'PER': 'personnes',
    'LOC': 'lieux',
    'ORG': 'organisations',
}


# ── Extraction from JSON ─────────────────────────────────────────────────

def normalize_entity_text(text):
    text = text.strip(' »«—.,;:!?()')
    tokens = text.split()
    deduped = []
    for tok in tokens:
        if not deduped or tok.lower() != deduped[-1].lower():
            deduped.append(tok)
    return ' '.join(deduped).strip()


def extract_entities_from_json(data):
    by_type = defaultdict(list)
    seen = defaultdict(set)

    for sentence in data.get('sentences', []):
        phrase = sentence.get('text', '').strip()
        for ent in sentence.get('entities', []):
            norm  = normalize_entity_text(ent.get('text', '').strip())
            etype = ent.get('type', 'UNK')
            if not norm or len(norm) < 2:
                continue
            if norm not in seen[etype]:
                seen[etype].add(norm)
                by_type[etype].append({
                    'entite': norm,
                    'type':   etype,
                    'phrase': phrase,
                })

    return by_type


# ── API DeepSeek ────────────────────────────────────────────────────────

def call_deepseek(prompt, retries=3):
    prompt = prompt.encode('utf-8', errors='replace').decode('utf-8')
    for attempt in range(retries):
        try:
            response = client.chat.completions.create(
                model="deepseek-chat",
                messages=[
                    {"role": "system", "content": DEDUP_SYSTEM},
                    {"role": "user",   "content": prompt},
                ],
                response_format={"type": "json_object"},
                temperature=0,
            )
            raw = response.choices[0].message.content.strip()
            raw = re.sub(r'^```json\s*', '', raw)
            raw = re.sub(r'\s*```$',     '', raw)
            return json.loads(raw)
        except json.JSONDecodeError:
            print(f"   ⚠ JSON invalide (tentative {attempt+1}/{retries})")
            time.sleep(2)
        except Exception as e:
            err = str(e)
            m = re.search(r'retry in ([\d.]+)s', err)
            wait = float(m.group(1)) + 2 if m else 5
            print(f"   ⚠ {err[:80]} — attente {wait:.0f}s")
            time.sleep(wait)
    return None


# ── Annotation by catégorie ──────────────────────────────────────────────────

def annotate_category(category, rows):
    """
    Envoie une catégorie à DeepSeek pour identifier les formes canoniques.
    Retourne les lignes originales inchangées + colonne nom_canonique ajoutée.
    Chaque forme originale reste sur sa propre ligne.
    """
    print(f"   {TYPE_LABELS.get(category, category):15s} : {len(rows)} entités → ", end='', flush=True)

    if len(rows) <= 1:
        row = dict(rows[0])
        row['nom_canonique'] = row['entite']
        print("1")
        return [row]

    entities_for_prompt = [r['entite'] for r in rows]
    prompt = DEDUP_PROMPT.format(
        category=TYPE_LABELS.get(category, category),
        entities=json.dumps(entities_for_prompt, ensure_ascii=False, indent=2)[:4000]
    )
    result = call_deepseek(prompt)

    if not result:
        print(f"{len(rows)} (échec API, nom_canonique = entite)")
        return [{**r, 'nom_canonique': r['entite']} for r in rows]

    # Construire un index : entite → nom_canonique
    canon_map = {}
    for ent in result.get('entites', []):
        canon     = ent.get('nom_canonique', '').strip()
        variantes = [v.strip() for v in ent.get('variantes', []) if v.strip()]
        if not canon:
            continue
        # Le canonique se mappe sur lui-même
        canon_map[canon] = canon
        # Chaque variante se mappe sur le canonique
        for v in variantes:
            canon_map[v] = canon

    # Annoter chaque ligne originale
    annotated = []
    for r in rows:
        row = dict(r)
        row['nom_canonique'] = canon_map.get(r['entite'], r['entite'])
        annotated.append(row)

    n_canonical = len(set(r['nom_canonique'] for r in annotated))
    print(f"{len(annotated)} lignes → {n_canonical} formes canoniques distinctes")
    time.sleep(1)
    return annotated


# ── Traitement principal ──────────────────────────────────────────────────────

def process_json(fname, content_bytes):
    stem = Path(fname).stem
    print(f"\n{'='*60}")
    print(f"Traitement : {fname}")
    print(f"{'='*60}")

    data      = json.loads(content_bytes.decode('utf-8', errors='replace'))
    sentences = data.get('sentences', [])
    print(f"→ {len(sentences)} phrases  |  résumé : {data.get('summary', {})}")

    print("\n→ Extraction des entités...")
    by_type = extract_entities_from_json(data)
    for etype, rows in by_type.items():
        print(f"   {TYPE_LABELS.get(etype, etype):15s} : {len(rows)}")

    print("\n→ Annotation via DeepSeek...")
    fieldnames = ['entite', 'nom_canonique', 'type', 'phrase']
    known      = set(TYPE_LABELS.keys())

    for etype, label in TYPE_LABELS.items():
        rows = by_type.get(etype, [])
        if not rows:
            continue
        annotated = annotate_category(etype, rows)
        out_csv   = f"{stem}_{label}_dedup.csv"
        with open(out_csv, 'w', encoding='utf-8-sig', newline='') as f:
            writer = csv.DictWriter(f, fieldnames=fieldnames, extrasaction='ignore')
            writer.writeheader()
            writer.writerows(annotated)
        print(f"   ✓ {out_csv}")
        files.download(out_csv)

    for etype, rows in by_type.items():
        if etype not in known and rows:
            annotated = annotate_category(etype, rows)
            out_csv   = f"{stem}_{etype}_dedup.csv"
            with open(out_csv, 'w', encoding='utf-8-sig', newline='') as f:
                writer = csv.DictWriter(f, fieldnames=fieldnames, extrasaction='ignore')
                writer.writeheader()
                writer.writerows(annotated)
            print(f"   ✓ {out_csv}")
            files.download(out_csv)


# ── Main ──────────────────────────────────────────────────────────────────────

print("\nSélectionnez un ou plusieurs fichiers .json (produits par l'outil NER slovène) :")
uploaded = files.upload()

for fname, content in uploaded.items():
    process_json(fname, content)

print("\n✓ Terminé.")


Clé API DeepSeek : ··········
✓ API configurée

Sélectionnez un ou plusieurs fichiers .json (produits par l'outil NER slovène) :


Saving merged.txt.ner.json to merged.txt.ner.json

Traitement : merged.txt.ner.json
→ 422 phrases  |  résumé : {'PER': 440, 'LOC': 391, 'ORG': 494}

→ Extraction des entités...
   organisations   : 203
   lieux           : 153
   personnes       : 211

→ Annotation via DeepSeek...
   personnes       : 211 entités → 211 lignes → 136 formes canoniques distinctes
   ✓ merged.txt.ner_personnes_dedup.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

   lieux           : 153 entités → 153 lignes → 95 formes canoniques distinctes
   ✓ merged.txt.ner_lieux_dedup.csv


...

<IPython.core.display.Javascript object>

   organisations   : 203 entités → 203 lignes → 99 formes canoniques distinctes
   ✓ merged.txt.ner_organisations_dedup.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


✓ Terminé.


In [ ]:
#check CSV

import pandas as pd
df=pd.read_csv('/content/C1merged_parsed.txt.ner_personnes_dedup.csv')

In [ ]:
df=df.drop(columns=['entite'])

In [ ]:
df=df.rename(columns={'nom_canonique': 'NAME'})

In [ ]:
df

,NAME,type,phrase
0,Đoko Kovačević,PER,# GLAVA I Đoko Kovačević. —
1,Đoko Kovačević,PER,Vijest o pogibiji na bojnom polju Španije Upo...
2,Đoko Kovačević,PER,"Tako sam i ja, pored zaduženja u samoj čelij, ..."
3,Đoko Kovačević,PER,To je učinilo da sam Đoka češće susretao — nek...
4,Đoko Kovačević,PER,Đokovim dolaskom dosegao je u našu čeliju novi...
5,Josip Broz Tito,PER,Đokovim dolaskom dosegao je u našu čeliju novi...
6,Đoko Kovačević,PER,Prilikom prvog susreta na sve nas je najsnažni...
7,Đoko Kovačević,PER,Puni značaj Đokova razgovora sa nama nije bio ...
8,Đoko Kovačević,PER,"Vjerovao sam Doku da ide po zadatku u Pariz, a..."
9,Đoko Kovačević,PER,Iako sam prilikom prvog susreta sa Đokom imao ...
